# Лабораторная работа №4. Акустическая модель (FastSpeech2)
Выполнила: Перминова Анастасия, группа M4221.

Обучены две модели FastSpeech2, различающиеся только входными фонемами: **gt** (фонемы из исходной разметки, запуск `gt_lr5e3`) и **g2p** (фонемы, предсказанные g2p из работы №3, запуск `g2p_lr5e3`). Итоговая модель для сдачи: **gt, `best_model_92926.pth`**.

## 1. Конфигурация, скорость обучения, время и вычислительные ресурсы


| | gt | g2p |
|---|---|---|
| Метаданные (train) | `metadata_phones_gt_train.csv` | `metadata_phones_train.csv` |
| pitch / energy | нет | нет |
| Batch | 16 | 16 |
| Номинальный lr | 5e-3, NoamLR (`warmup_steps=4000`) | 5e-3, NoamLR |
| Эпох | 100 | 100 |
| Лучший чекпоинт | `best_model_92926` | `best_model_96759` |
| Время | 18.11 02:42 – 19.11 05:12, **≈26,5 ч** | 19.11 11:33 – 20.11 16:46, ≈29 ч, из них ≈5 ч простоя, **≈24 ч** чистого обучения |
| Ресурсы | NVIDIA GeForce RTX 3050 Laptop 6 ГБ, Windows, PyTorch 2.5.1+cu121, fp32 | то же |

Данные: 15 779 обучающих фраз (157 (1%) из которых откладывается для eval-loss), тест: строки с номером, делящимся на 5 (120 фраз в `eval_sentences.csv`). Скорость ≈10 мин на эпоху.

Выбор lr и обоснование. Взят номинальный lr 5e-3 как самый быстрый вариант. Но расписание NoamLR в запуске шагает раз в эпоху (`scheduler_after_epoch=true`), поэтому за 100 эпох lr вырос лишь от 1,25e-6 до ≈1,25e-4, то есть реально обучение шло на уровне значения по умолчанию в coqui (1e-4).

Стандартное расписание было сохранено, потому что первый запуск с пошаговым ростом lr (`baseline_gt`, lr 1e-3, `scheduler_after_epoch=False`) развалился...

*Ограничение по ресурсам:* 6 ГБ памяти не хватает для batch 32, на который был рассчитан скрипт, поэтому взят batch 16. Запуск g2p прервался на 13й эпохе ошибкой CUDA и был продолжен из `best_model_12454.pth`.

![lr](img/gt_lr.png) ![lr g2p](img/g2p_lr.png)

## 2. Кривые loss
Синий цвет: gt, красный: g2p (tensorboard).

**Loss спектра** (eval и train):

![loss_spec gt](img/gt_loss_spec.png)
![loss_spec g2p](img/g2p_loss_spec.png)

**Loss предсказателя длительностей и ошибка длительностей в кадрах** (eval и train):

![dur gt](img/gt_dur.png)
![dur g2p](img/g2p_dur.png)

**Норма градиента:**

![grad gt](img/gt_grad_norm.png) ![grad g2p](img/g2p_grad_norm.png)

Наблюдения:
* До шага ≈10k loss спектра стоит на плато ≈3,5, а ошибка длительностей ≈5,5 кадров: aligner еще не сопоставил текст и аудио. Затем loss спектра резко падает, а ошибка длительностей снижается до ≈2 кадров (aligner "нашел" выравнивание).
* `loss_dur` после этого момента подскакивает (≈0,012 → 0,037) и плавно убывает до ≈0,014 (eval) и ≈0,009 (train). Вероятно, что, пока выравнивание плохое, цели почти равномерные и их легко предсказывать, но потом они становятся сложнее и непредсказуемее: aligner находит настоящее соответствие текста и звука, и теперь длительности разные: гласные длиннее согласных, паузы длиннее обычных фонем.
* Eval loss спектра выходит на плато ≈0,8 с ~30k шагов, train падает до ≈0,45 (переобучение); у gt после ~70k шагов eval слегка растет (≈0,78 → 0,9).
* Градиенты стабильны (норма ≈2-2,5 при клиппинге 5), NaN и взрывов нет.
* Кривые gt и g2p почти совпадают. Финальный eval loss: gt 2,41, g2p 2,37; отложенный 1% у запусков разный, поэтому разницу нельзя считать "преимуществом" g2p.

## 3. Замеры на 120 отложенных фразах
(`evaluate.py`, чекпоинты gt: `best_model_92926`, g2p: `best_model_96759`.)

| | gt | g2p |
|---|---|---|
| Фраз синтезировано | 120 | 120 |
| MCD модели, среднее, дБ | **28.83** (std 3.04) | **28.82** (std 2.79) |
| MCD модели, медиана, дБ | 28.22 | 28.43 |
| MCD постоянного мела, дБ | 62.09 | 62.09 |
| Фраз, где модель лучше постоянного мела | 120 из 120 | 120 из 120 |
| Отношение числа кадров model/natural, среднее | 0.991 | 0.961 |
| Диапазон мела, настоящий | [−11.33; 0.48] | [−11.33; 0.48] |
| Диапазон мела, модель | [−11.23; −0.51] | [−11.10; −0.53] |

MCD у gt и g2p практически одинаков (разница 0,01 дБ). Есть одно различие: у g2p синтез в среднем на ≈4% короче оригинала, то есть темп чуть быстрее.
MCD модели 28,8 дБ - в 2,2 раза лучше значения постоянного мела на всех 120 фразах.

## 4. Акустическое сравнение natural / copysynth / model

* **natural** - настоящая запись, эталон, речь звучит естественно, "живо".
* **copysynth** - звучит более "глухо", слышен металлический призвук, видимо, из-за потери деталей при переходе к 80-полосному мелу и особенностей восстановления фазы Griffin-Lim. Это максимальное качество звука для данной модели.
* **model** - слова понятны, речь разобрать можно, но звук напоминает разговор "в вентилятор": шумный, дребезжащий, "терминаторский" голос. Причины могут быть такие: предсказанный мел сглажен, гармоническая структура размыта, Griffin-Lim плохо восстанавливает фазу по такому мелу. Также  pitch и energy не использовались, что также усиливает усреднение и монотонность, размывает гармоники.

### Пример:

In [27]:
from pathlib import Path
from IPython.display import Audio, HTML, display
import base64

EVAL = Path("runs/gt_lr5e3/eval_best_full")
ids = ["000020_RUSLAN"]
kinds = ["natural", "copysynth", "model"]

def player(path):
    b64 = base64.b64encode(Path(path).read_bytes()).decode()
    return f'<audio controls src="data:audio/wav;base64,{b64}" style="width:260px"></audio>'

rows = ""
for i in ids:
    cells = "".join(f"<td style='padding:4px 10px'>{player(EVAL/k/f'{i}.wav')}</td>" for k in kinds)
    rows += f"<tr><td><b>{i}</b></td>{cells}</tr>"

header = "".join(f"<th>{k}</th>" for k in kinds)
display(HTML(f"<table><tr><th>id</th>{header}</tr>{rows}</table>"))

id,natural,copysynth,model
000020_RUSLAN,,,


## 5. Оценка фонетического и мелодического качества
Слова разборчивы, основная проблема - "дребезщащий" звук, а не ошибки фонем. Речь монотонная, без интонаций. Нельзя сказать, что gt или g2p сильно различаются по качеству. В разных примерах то одна, то другая по ощущениям звучит лучше. Например, в записи `003660` gt кажется громче, гласные как будто более растянуты, отчего и "железо" в голосе слышится более явно. В g2p звук тише, звуки более короткие, поэтому кажется, что речь чище, более ровная:

In [28]:
RUN_DIRS = {"gt": Path("runs/gt_lr5e3/eval_best_full"), "g2p": Path("runs/g2p_lr5e3/eval_best_full")}

def _player(path):
    b64 = base64.b64encode(Path(path).read_bytes()).decode()
    return f'<audio controls src="data:audio/wav;base64,{b64}" style="width:280px"></audio>'

def show(utt_id):
    cells = "".join(
        f"<td style='padding:4px 10px'><b>model ({k})</b><br>{_player(d / 'model' / f'{utt_id}.wav')}</td>"
        for k, d in RUN_DIRS.items()
    )
    display(HTML(f"<h4>{utt_id}</h4><table><tr>{cells}</tr></table>"))

show("003660_RUSLAN")

model (gt),model (g2p)


В следующем примере оба произношения спотыкаются на слове "только", а g2p делает акцент на неударной "о" в слове "пошляки", поэтому оно звучит неестественно:

In [29]:
show("005505_RUSLAN")

model (gt),model (g2p)


Проблема с ударением в обоих вариантах: гОрла вместо горлА:

In [30]:
show("007155_RUSLAN")

model (gt),model (g2p)


В примере ниже у g2p ударение в последнем слове звучит правильнее: алексАндрова, а не александровА, как у gt, но мне кажется, это связано с тем, что сама гласная "а" у g2p короче, поэтому ударность просто не успевает проявиться.

In [31]:
show("021115_RUSLAN")

model (gt),model (g2p)


А вот тут g2p выделяет предлог "на", и это неправильно звучит:

In [32]:
show("006175_RUSLAN")

model (gt),model (g2p)


Здесь g2p словно лучше ловит вопросительную интонацию: "Текст?". Но это единичный случай попадания. Вообще заметила, что у gt интонации больше нисходящие, а у gtp восходящие, тут это сыграло на руку.

In [33]:
show("008010_RUSLAN")

model (gt),model (g2p)


Отдельным наблюдением можно отметить сложности с произнесением звука "л". Он зачастую проглатывается обеими моделями.

## 6. Что осталось неудачным и гипотезы
1. Первый запуск `baseline_gt` не сошелся.
Запуск шел с lr 1e-3 и scheduler_after_epoch=False, то есть lr рос по шагам и достигал пика около шага 4000. За ≈60k шагов eval loss спектра сначала дошел до минимума ≈3,76 (около шага 5k), а затем вырос до ≈4,3, а train стоял на плато ≈3,3. Оценка на 5 тестовых фразах показала MCD 58,1–58,8 дБ при 56,8 дБ у постоянного мела, то есть модель не лучше варианта «ничего не предсказывать». Максимум предсказанного мела был около −3 при настоящем +0,55, на слух получался сплошной гул. При этом длительности выучены (отношение кадров ≈0,89): данные, разметка и выравнивание в порядке, не работал именно предсказатель спектра. Возможно, все сломал слишком быстрый рост lr при батче 16: модель стала выдавать одно усредненное по корпусу значение вместо связанного с фонемами. Для основных запусков оставлено стандартное расписание по эпохам.

   ![baseline eval loss](img/baseline_gt_loss_spec.png)

2. Железный звук, сглаженный мел.
В model слова разобрать можно, но звук напоминает шумную металлическую роботизированную речь: дребезжащий оттенок вместо чистых гармоник голоса. Диапазон мела подтверждает срезанные пики: нижняя граница совпадает с настоящей (−11,23 у gt и −11,10 у g2p против −11,33), а верхняя ниже примерно на 1 (−0,51 и −0,53 против +0,48). Размах модели ≈10,7 и 10,6 против 11,8. copysynth при этом звучит приемлемо, значит, основной вклад дают не Griffin-Lim, а акустическая модель.
Модель без предсказателей высоты тона и энергии (use_pitch=False, use_energy=False), обученная по MSE, предсказывает усредненный, сглаженный мел, теряя тонкую гармоническую структуру, а Griffin-Lim превращает такой мел в шумовую текстуру.

3. Запоминание корпуса.
Eval loss спектра вышел на плато ≈0,8 уже с ~30k шагов, а train продолжал падать до ≈0,45. У gt после ~70k шагов eval слегка вырос (≈0,78 → 0,9). Разрыв train/eval показывает, что часть улучшений дает запоминание обучающих фраз, а не обобщение - дальнейшие эпохи почти не помогали. Ориентиром служит то, что checkpoint_60000 и best_model_92926 на пробных фразах дали одинаковый MCD. Однако eval-часть всего 157 фраз, поэтому колебания в десятые могут быть шумом.

4. Точность длительностей.
Ошибка предсказателя длительностей составила ≈1,9 кадра на фонему (≈22 мс). По длине фраз gt близка к оригиналу (0,991), а g2p в среднем на ≈4% короче (0,961), то есть говорит чуть быстрее. Внутренний aligner дает приблизительные цели, и более точные внешние длительности из разметки MFA, наверное, улучшили бы темп и ритм.

5. Ограничение вокодера Griffin-Lim.
Немного металлический, приглушенный призвук слышен уже в copysynth, то есть он не связан с моделью. Это максимум качества данного способа восстановления звука.